# SwingScope on Google Colab

County clustering, swing detection, and calibrated win probability.

Runs end to end on a free CPU instance in about 3-5 minutes. No Kaggle key needed in synthetic mode.


## 1. Setup

Upload `swingscope.zip` using the file browser on the left, then run this cell.


In [ ]:
import os, sys, zipfile, pathlib

ZIP = '/content/swingscope.zip'
ROOT = '/content/swingscope'

if os.path.exists(ZIP) and not os.path.exists(ROOT):
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall('/content')

if os.path.exists(ROOT):
    os.chdir(ROOT)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('cwd:', os.getcwd())
print(sorted(p.name for p in pathlib.Path('.').iterdir()))


## 2. Dependencies

Colab already ships torch, scikit-learn, pandas, plotly and matplotlib. This only fills gaps.


In [ ]:
import importlib

missing = [m for m in ['torch', 'sklearn', 'pandas', 'numpy', 'plotly', 'matplotlib', 'pyarrow', 'joblib'] if importlib.util.find_spec(m) is None]
print('missing:', missing)

if missing:
    mapping = {'sklearn': 'scikit-learn'}
    pkgs = ' '.join(mapping.get(m, m) for m in missing)
    !pip install -q {pkgs}

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())


## 3. Run the whole pipeline

Switch `--mode synthetic` to `--mode real` after downloading the datasets.


In [ ]:
!python -m src.run_all --mode synthetic --epochs 150 --ablation


## 4. Module A: county archetypes


In [ ]:
import json
import pandas as pd
from src.config import CLUSTER_PATH, REPORT_DIR
from src.io_utils import load_table

clusters = load_table(CLUSTER_PATH)
diag = json.load(open(REPORT_DIR / 'cluster_diagnostics.json'))

print('best k:', diag['best_k'])
print('explained variance:', [round(v, 3) for v in diag['explained_variance_ratio']])
print('dbscan outliers:', diag['dbscan_outliers'])

pd.read_csv(REPORT_DIR / 'cluster_profile.csv')


In [ ]:
from src.viz.pca_plot import pca_scatter
from IPython.display import Image, display

display(Image(filename=str(pca_scatter(clusters))))


## 5. Module B: swing detection


In [ ]:
metrics = json.load(open(REPORT_DIR / 'module_b_metrics.json'))

print('test macro-F1 :', round(metrics['metrics']['test_macro_f1'], 4))
print('test accuracy :', round(metrics['metrics']['test_accuracy'], 4))
print()
print('baselines')
for name, vals in metrics['baselines'].items():
    print(f'  {name:<26} {vals["macro_f1"]}')

if 'ablation' in metrics:
    print()
    print('ablation')
    for name, vals in metrics['ablation'].items():
        print(f'  {name:<28} {round(vals["test_macro_f1"], 4)}')


In [ ]:
from src.evaluate import confusion_plot

display(Image(filename=str(confusion_plot(metrics['confusion_matrix']))))


## 6. The volatility map

Interactive choropleth. Falls back to a bar chart if the GeoJSON fetch is blocked.


In [ ]:
from src.config import SWING_PRED_PATH
from src.viz.volatility_map import choropleth, top_volatile_table

swing = load_table(SWING_PRED_PATH)
result = choropleth(swing)
print(result)

top_volatile_table(swing)


In [ ]:
import plotly.express as px
from src.viz.volatility_map import load_geojson

try:
    counties = load_geojson()
    fig = px.choropleth(
        swing,
        geojson=counties,
        locations='fips',
        color='p_swing',
        color_continuous_scale='magma',
        range_color=(0, 1),
        scope='usa',
        labels={'p_swing': 'P(Swing)'},
    )
    fig.update_traces(marker_line_width=0)
    fig.update_layout(title='Most Volatile Counties - SwingScope', margin=dict(l=0, r=0, t=50, b=0))
    fig.show()
except Exception as exc:
    print('choropleth unavailable:', exc)


## 7. Module C: calibrated win probability


In [ ]:
from src.evaluate import reliability_plot

mc = json.load(open(REPORT_DIR / 'module_c_metrics.json'))
print('brier (calibrated) :', round(mc['calibrated']['brier'], 4))
print('brier (raw)        :', round(mc['raw']['brier'], 4))
print('brier (always 0.5) :', round(mc['baseline_brier_always_half'], 4))
print('auc                :', round(mc['calibrated']['auc'], 4))
print('synthetic spend    :', mc['spend_is_synthetic'])

display(Image(filename=str(reliability_plot(mc['reliability']))))


## 8. Score a hypothetical candidate


In [ ]:
import joblib
import numpy as np
import torch
from src.config import MODEL_DIR, RACE_PATH
from src.module_c_winprob import WinProbNet

ckpt = torch.load(MODEL_DIR / 'winprobnet.pt', map_location='cpu', weights_only=False)
features = ckpt['features']
model = WinProbNet(len(features))
model.load_state_dict(ckpt['state_dict'])
model.eval()

scaler = joblib.load(MODEL_DIR / 'winprob_scaler.joblib')
calibrator = joblib.load(MODEL_DIR / 'winprob_calibrator.joblib')
races = load_table(RACE_PATH)


def win_probability(is_incumbent=1, prev_margin=0.04, prev_margin_2=0.02, p_swing=0.7,
                    log_spend=14.0, log_spend_ratio=0.0, national_env=0.02, cluster=None):
    row = {f: 0.0 for f in features}
    row.update({
        'is_incumbent': float(is_incumbent),
        'prev_margin': prev_margin,
        'prev_margin_2': prev_margin_2,
        'p_swing': p_swing,
        'log_spend': log_spend,
        'log_spend_ratio': log_spend_ratio,
        'national_env': national_env,
        'partisan_lean': prev_margin - float(races.prev_margin.mean()),
    })
    if cluster and f'cluster_mix_{cluster}' in row:
        row[f'cluster_mix_{cluster}'] = 1.0
    X = scaler.transform(pd.DataFrame([row])[features].astype(float))
    with torch.no_grad():
        raw = float(torch.sigmoid(model(torch.tensor(X, dtype=torch.float32)))[0])
    return float(calibrator.predict([raw])[0])


print('incumbent, +4 margin  :', round(win_probability(1, 0.04) * 100, 1), '%')
print('challenger, +4 margin :', round(win_probability(0, 0.04) * 100, 1), '%')
print('incumbent, -6 margin  :', round(win_probability(1, -0.06) * 100, 1), '%')
print('outspent 5x, +2       :', round(win_probability(1, 0.02, log_spend_ratio=-1.6) * 100, 1), '%')


## 9. Feature attribution


In [ ]:
try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

from src.config import TEST_YEAR

test = races[races.year == TEST_YEAR]
X_test = scaler.transform(test[features].astype(float))
background = torch.tensor(X_test[:200], dtype=torch.float32)
sample = torch.tensor(X_test[200:600], dtype=torch.float32)

explainer = shap.DeepExplainer(model, background)
values = explainer.shap_values(sample, check_additivity=False)
shap.summary_plot(values, sample.numpy(), feature_names=features, max_display=15)


## 10. Launch the demo app

Use this for the demo video. The cell prints a public URL.


In [ ]:
!pip install -q streamlit
!npm install -g localtunnel 2>/dev/null | tail -1
!curl -s https://loca.lt/mytunnelpassword
!streamlit run app/streamlit_app.py --server.port 8501 &>/content/streamlit.log &
!sleep 6 && npx localtunnel --port 8501


## 11. Save results back to Drive


In [ ]:
from google.colab import files
import shutil

shutil.make_archive('/content/swingscope_results', 'zip', 'reports')
files.download('/content/swingscope_results.zip')
